# Глава 6. Регуляризация

Код к главе 6 книги 1: гребневая регрессия на примере с почти совпадающими признаками,
сингулярное разложение, LASSO и обнуление весов, пути регуляризации, подбор $\lambda$
кросс-валидацией (листинг 6.1) и сокращение весов в градиентном спуске.

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markpolyak/ml-textbook-code/blob/main/book1/ch06_regularization.ipynb)

In [ ]:
# В Colab ставим пакет mlbook с данными книги (локально: pip install -e . в корне репозитория)
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/markpolyak/ml-textbook-code

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mlbook import flats

## Гребневая регрессия

Пример из гл. 3: строки $\mathbf{F}$ — $(1;\ 1)$, $(2;\ 2)$, $(3;\ 3{,}1)$, свободного члена
нет. Решение $\boldsymbol{\theta}^*_\lambda = (\mathbf{F}^{\mathrm{T}}\mathbf{F} +
\lambda\mathbf{I})^{-1}\mathbf{F}^{\mathrm{T}}\mathbf{y}$ при трёх значениях $y_3$ (табл. 6.1).

In [ ]:
F = np.array([[1, 1], [2, 2], [3, 3.1]])


def ridge_solve(F, y, lam):
    return np.linalg.solve(F.T @ F + lam * np.eye(F.shape[1]), F.T @ y)


print("определитель FᵀF + 0,1·I:", round(np.linalg.det(F.T @ F + 0.1 * np.eye(2)), 3))
for lam in (0, 0.1, 1):
    rows = []
    for y3 in (6.0, 6.1, 6.2):
        th = ridge_solve(F, np.array([2, 4, y3]), lam)
        rows.append(f"{th.round(2)} → {np.array([2, 3]) @ th:.2f}")
    print(f"λ = {lam}:", ";  ".join(rows))

Цена устойчивости — смещение: с ростом $\lambda$ растёт ошибка на обучающей выборке,
прогнозы занижаются.

In [ ]:
y = np.array([2, 4, 6.1])
for lam in (0, 0.1, 1):
    th = ridge_solve(F, y, lam)
    print(f"λ = {lam}: MSE на обучении {np.mean((F @ th - y) ** 2):.4f}, прогноз для x3 {F[2] @ th:.2f}")

**Сингулярное разложение.** Множитель $1/s_j$ заменяется на $s_j/(s_j^2 + \lambda)$:
неустойчивое направление с малым $s_2$ подавляется. Число обусловленности матрицы
системы падает с ≈ 16 400 до ≈ 280.

In [ ]:
V, s, Ut = np.linalg.svd(F, full_matrices=False)
lam = 0.1
print("s =", s.round(3))
print("множители МНК 1/s:", (1 / s).round(2), " гребня s/(s² + λ):", (s / (s ** 2 + lam)).round(2))
print(f"κ: {s[0] ** 2 / s[1] ** 2:.0f} → {(s[0] ** 2 + lam) / (s[1] ** 2 + lam):.0f}")
print("решение через SVD:", (Ut.T @ (s / (s ** 2 + lam) * (V.T @ y))).round(4),
      " напрямую:", ridge_solve(F, y, lam).round(4))

## LASSO

При ортонормированных признаках веса находятся по отдельности: гребневая регрессия
сжимает решение МНК $a_j$ в $1 + \lambda$ раз, LASSO сдвигает его к нулю на $\lambda/2$
и обнуляет малые веса (задача 2).

In [ ]:
a = np.array([3, -0.5, 1.2])
lam = 2
print("гребень:", (a / (1 + lam)).round(2))
print("LASSO:  ", np.sign(a) * np.maximum(np.abs(a) - lam / 2, 0) + 0.0)   # + 0.0: без «−0»

Проверим формулу на настоящих ортонормированных признаках с классом `Lasso`. Его
функционал — $\frac{1}{2\ell}\|\mathbf{F}\boldsymbol{\theta} - \mathbf{y}\|^2 +
\alpha\|\boldsymbol{\theta}\|_1$, поэтому $\lambda$ книги равно $2\ell\alpha$.

In [ ]:
from sklearn.linear_model import Lasso

rng = np.random.default_rng(0)
Fo = np.linalg.qr(rng.normal(size=(50, 3)))[0]          # 50 × 3, столбцы ортонормированы
yo = Fo @ a + (np.eye(50) - Fo @ Fo.T) @ rng.normal(size=50)   # решение МНК — ровно a
print("МНК:", np.linalg.lstsq(Fo, yo, rcond=None)[0].round(6))
print("Lasso:", Lasso(alpha=lam / (2 * 50), fit_intercept=False).fit(Fo, yo).coef_.round(4) + 0.0)

Один вес: минимум $(\theta - 2)^2 + \lambda\theta^2$ равен $2/(1 + \lambda)$, минимум
$(\theta - 2)^2 + \lambda|\theta|$ — $\max(2 - \lambda/2,\ 0)$ (рис. 6.3).

In [ ]:
t = np.linspace(-1, 3, 400)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for lam in (0, 1, 3):
    q = (t - 2) ** 2 + lam * t ** 2
    a1.plot(t, q, label=f"λ = {lam}")
    a1.plot(2 / (1 + lam), np.min(q), "o", color=a1.lines[-1].get_color())
for lam in (0, 2, 4):
    q = (t - 2) ** 2 + lam * np.abs(t)
    a2.plot(t, q, label=f"λ = {lam}")
    a2.plot(max(2 - lam / 2, 0), np.min(q), "o", color=a2.lines[-1].get_color())
a1.set_title(r"а) гребень: $(\theta - 2)^2 + \lambda\theta^2$")
a2.set_title(r"б) LASSO: $(\theta - 2)^2 + \lambda|\theta|$")
for ax in (a1, a2):
    ax.set_ylim(0, 10)
    ax.axvline(0, color="0.7", linewidth=0.8)
    ax.set_xlabel(r"$\theta$")
    ax.legend()
plt.show()

## Геометрия: круг и ромб

Ищем минимум квадратичной функции с центром в решении МНК $(2;\ 1)$ на круге
$\theta_1^2 + \theta_2^2 \le 1$ и на ромбе $|\theta_1| + |\theta_2| \le 1$ — перебором точек
сетки (рис. 6.1). На ромбе решение попадает в вершину, где $\theta_2 = 0$.

In [ ]:
c = np.array([2.0, 1.0])
A = np.array([[1.4, 0.5], [0.5, 1.0]])
T1, T2 = np.meshgrid(np.linspace(-1.8, 3.2, 400), np.linspace(-1.8, 3.2, 400))
D = np.stack([T1 - c[0], T2 - c[1]], axis=-1)
Qg = np.einsum("...i,ij,...j->...", D, A, D)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, name, inside in [(axes[0], "L2", T1 ** 2 + T2 ** 2 <= 1),
                         (axes[1], "L1", np.abs(T1) + np.abs(T2) <= 1)]:
    k = np.argmin(np.where(inside, Qg, np.inf))
    p = (T1.flat[k], T2.flat[k])
    print(name, "решение:", np.round(p, 2))
    ax.contour(T1, T2, Qg, levels=[Qg.flat[k] * m for m in (0.25, 0.6, 1, 1.6)], colors="C0")
    ax.contourf(T1, T2, inside, levels=[0.5, 1], colors=["#dbe6f4"])
    ax.plot(*c, "o", color="C0")
    ax.plot(*p, "o", color="C1")
    ax.axhline(0, color="0.6", linewidth=0.6)
    ax.axvline(0, color="0.6", linewidth=0.6)
    ax.set_aspect("equal")
    ax.set_title(name)
plt.show()

## Пути регуляризации

Синтетические квартиры с пятью стандартизованными признаками: площадь, жилая площадь
(почти копия площади), расстояние до метро, этаж и случайный признак; цена зависит только
от площади и метро (рис. 6.2). Данные те же, что на рисунке в книге.

In [ ]:
from sklearn.linear_model import Ridge

rng = np.random.default_rng(5)
d = flats.make(n=300, seed=4)
area, metro, price = d["area"], d["metro"], d["price"]
living = 0.62 * area + rng.normal(0, 2.0, area.size)       # жилая площадь — почти копия площади
floor = rng.integers(1, 17, area.size).astype(float)       # этаж: на цену не влияет
noise = rng.normal(size=area.size)                         # случайный признак
X = np.column_stack([area, living, metro, floor, noise])
names = ["площадь", "жилая площадь", "до метро", "этаж", "случайный"]
Xs = (X - X.mean(0)) / X.std(0)
yc = price - price.mean()
print("корреляция площади и жилой площади:", np.corrcoef(area, living)[0, 1].round(3))
print("МНК:", dict(zip(names, np.linalg.lstsq(Xs, yc, rcond=None)[0].round(2).tolist())))

In [ ]:
lams = np.logspace(-1, 4, 60)
Wr = np.array([Ridge(alpha=lam).fit(Xs, yc).coef_ for lam in lams])
Wl = np.array([Lasso(alpha=lam / (2 * len(yc)), max_iter=50_000).fit(Xs, yc).coef_
               for lam in lams])

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, W, title in [(axes[0], Wr, "а) гребневая регрессия"), (axes[1], Wl, "б) LASSO")]:
    for j, nm in enumerate(names):
        ax.plot(lams, W[:, j], label=nm)
    ax.axhline(0, color="0.6", linewidth=0.6)
    ax.set_xscale("log")
    ax.set_xlabel(r"$\lambda$")
    ax.set_title(title)
axes[0].set_ylabel("вес признака")
axes[1].legend()
plt.show()

В каком порядке LASSO обнуляет веса — первое $\lambda$ сетки, при котором вес стал нулём:

In [ ]:
for j in np.argsort([lams[np.argmax(Wl[:, j] == 0)] for j in range(5)]):
    print(f"{names[j]:14} λ ≈ {lams[np.argmax(Wl[:, j] == 0)]:.3g}")

## Подбор $\lambda$ кросс-валидацией

Листинг 6.1 на тех же пяти признаках (в листинге признаки стандартизует конвейер).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, price, test_size=0.25, random_state=0)

In [ ]:
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ridge = make_pipeline(StandardScaler(),
                      RidgeCV(alphas=np.logspace(-3, 3, 13)))
lasso = make_pipeline(StandardScaler(), LassoCV(cv=5))
ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)
print(ridge[-1].alpha_)        # выбранный коэффициент
# сколько признаков осталось в модели LASSO
print(np.sum(lasso[-1].coef_ != 0))

In [ ]:
from sklearn.metrics import mean_squared_error

print("веса LASSO:", dict(zip(names, (lasso[-1].coef_.round(3) + 0.0).tolist())))
for name, m in [("гребень", ridge), ("LASSO", lasso)]:
    print(f"{name}: MSE на тесте {mean_squared_error(y_test, m.predict(X_test)):.3f}")

## Сокращение весов в градиентном спуске

Шаг $\boldsymbol{\theta} := (1 - 2\eta\lambda)\,\boldsymbol{\theta} - \eta\,\nabla
\|\mathbf{F}\boldsymbol{\theta} - \mathbf{y}\|^2$ приходит в решение гребневой регрессии.

In [ ]:
y = np.array([2, 4, 6.1])
theta, eta, lam = np.zeros(2), 0.005, 1.0
for _ in range(20_000):
    theta = (1 - 2 * eta * lam) * theta - eta * 2 * F.T @ (F @ theta - y)
print("спуск:", theta.round(4), " формула:", ridge_solve(F, y, lam).round(4))

## Задачи

**1.** Гребневая регрессия при $\lambda = 1$, $\mathbf{y} = (2;\ 4;\ 6{,}1)$.

In [ ]:
M = F.T @ F + np.eye(2)
print(M, " определитель", round(np.linalg.det(M), 2), " θ =", ridge_solve(F, y, 1).round(3))

**3.** Признак умножили на 10: вес МНК уменьшается ровно в 10 раз, а у гребневой регрессии
меняется и модель.

In [ ]:
F10 = F * [10, 1]
print("МНК:", np.linalg.lstsq(F, y, rcond=None)[0].round(4), "→",
      np.linalg.lstsq(F10, y, rcond=None)[0].round(4))
print("гребень:", ridge_solve(F, y, 1).round(4), "→", ridge_solve(F10, y, 1).round(4))